# Project 4: Listening Past the Noise — Submission 4
### Classical Acoustic Ensembling on 77-Dimensional Spectral & Timbral Descriptors

**Architecture Overview:**
- **Feature Engineering (77 Dimensions):**
  1. **MFCCs (28 features):** 14 cepstral coefficients (mean + standard deviation across time).
  2. **Chroma STFT (24 features):** 12 pitch classes (mean + standard deviation).
  3. **Spectral Descriptors (11 features):** Spectral Centroid, Bandwidth, Rolloff, Zero Crossing Rate, RMS energy (means + stds), and total audio duration.
  4. **Spectral Contrast (14 features):** Octave-band energy contrasts across 7 frequency sub-bands (mean + std).
- **4-Model Ensemble:**
  * **40% Support Vector Classifier (SVC):** RBF kernel, $C=5.0$, $\gamma=0.01$, balanced class weights.
  * **10% Logistic Regression:** $L_2$ regularization, $C=1.0$, balanced class weights.
  * **30% Random Forest:** 500 decision trees, square-root feature subsampling.
  * **20% Extra Trees:** 500 extremely randomized trees.
- **Validation & Leaderboard:**
  * Trained on all verified recordings ($N=620$, 41 classes).
  * Fast inference: extracts all features and predicts in under 2 minutes.

In [ ]:
import os, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import soundfile as sf, librosa
warnings.filterwarnings("ignore")

# Primary dataset path: Kaggle competition input directory
DATA_DIR = Path("/kaggle/input/competitions/project4")
if not DATA_DIR.exists():
    LOCAL_FALLBACK = Path("/home/pranav/WORK/Work/IIITD/ML/Kaggle_comp/project4")
    if LOCAL_FALLBACK.exists():
        DATA_DIR = LOCAL_FALLBACK
    else:
        matches = list(Path("/kaggle/input").rglob("train.csv"))
        DATA_DIR = matches[0].parent if matches else Path("project4")

# Working directory for Kaggle output / caching
WORK = Path("/kaggle/working")
if not WORK.exists():
    WORK = Path(".")

# Load dataset metadata
train_df  = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
test_df   = pd.read_csv(DATA_DIR / "test.csv",  dtype={"id": str})
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str})

# Audio directory paths
if (DATA_DIR / "audio/audio/train").exists():
    TRAIN_AUDIO = DATA_DIR / "audio/audio/train"
    TEST_AUDIO  = DATA_DIR / "audio/audio/test"
elif (DATA_DIR / "audio/train").exists():
    TRAIN_AUDIO = DATA_DIR / "audio/train"
    TEST_AUDIO  = DATA_DIR / "audio/test"
else:
    train_audio_matches = [p for p in DATA_DIR.rglob("train") if p.is_dir()]
    test_audio_matches  = [p for p in DATA_DIR.rglob("test") if p.is_dir()]
    TRAIN_AUDIO = train_audio_matches[0] if train_audio_matches else DATA_DIR / "audio/train"
    TEST_AUDIO  = test_audio_matches[0] if test_audio_matches else DATA_DIR / "audio/test"

print(f"Data directory: {DATA_DIR}")
print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")
print(f"Train audio: {TRAIN_AUDIO} | Test audio: {TEST_AUDIO}")

from joblib import Parallel, delayed
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

verified_df = train_df[train_df["checked"] == 1].reset_index(drop=True)
v_ids = verified_df["id"].tolist()
y_train = verified_df["label"].to_numpy()
test_ids = test_df["id"].tolist()

classes = np.sort(np.unique(y_train))
n_classes = len(classes)
print(f"Verified training recordings: {len(v_ids)} across {n_classes} classes")
print(f"Test recordings to evaluate: {len(test_ids)}")

## 1. 77-Dimensional Audio Feature Extraction

In [ ]:
def extract_77_features(file_path):
    try:
        y, sr = sf.read(str(file_path))
        if y.ndim > 1:
            y = np.mean(y, axis=1)
        y = y.astype(np.float32)

        # 1. MFCC (14 coefficients: 14 mean + 14 std = 28 features)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=14, n_fft=2048, hop_length=512)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)

        # 2. Chroma STFT (12 mean + 12 std = 24 features)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=2048, hop_length=512)
        chroma_mean = np.mean(chroma, axis=1)
        chroma_std = np.std(chroma, axis=1)

        # 3. Spectral Statistics + Duration (11 features)
        cent = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=2048, hop_length=512)
        bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=2048, hop_length=512)
        ro = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=2048, hop_length=512)
        zcr = librosa.feature.zero_crossing_rate(y=y, frame_length=2048, hop_length=512)
        rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)
        duration = len(y) / sr

        spectral_stats = np.array([
            np.mean(cent), np.std(cent),
            np.mean(bw), np.std(bw),
            np.mean(ro), np.std(ro),
            np.mean(zcr), np.std(zcr),
            np.mean(rms), np.std(rms),
            duration
        ], dtype=np.float32)

        # 4. Spectral Contrast (7 bands: 7 mean + 7 std = 14 features)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=2048, hop_length=512)
        contrast_mean = np.mean(contrast, axis=1)
        contrast_std = np.std(contrast, axis=1)

        return np.concatenate([
            mfcc_mean, mfcc_std,
            chroma_mean, chroma_std,
            spectral_stats,
            contrast_mean, contrast_std
        ])
    except Exception as e:
        return np.zeros(77, dtype=np.float32)

print("Extracting features in parallel across CPU cores...")
t0 = time.time()
X_train = np.array(Parallel(n_jobs=-1, batch_size=16, verbose=1)(
    delayed(extract_77_features)(TRAIN_AUDIO / f"{fid}.wav") for fid in v_ids
), dtype=np.float32)

X_test = np.array(Parallel(n_jobs=-1, batch_size=16, verbose=1)(
    delayed(extract_77_features)(TEST_AUDIO / f"{fid}.wav") for fid in test_ids
), dtype=np.float32)

print(f"Features extracted in {time.time()-t0:.1f}s")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
assert X_train.shape[1] == 77 and X_test.shape[1] == 77

## 2. Model Training & 4-Model Soft Ensembling

In [ ]:
# Define winning ensemble configurations
models = {
    "SVM": make_pipeline(
        StandardScaler(),
        SVC(C=5.0, kernel="rbf", gamma=0.01, class_weight="balanced", probability=True, random_state=42)
    ),
    "LR": make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000, random_state=42)
    ),
    "RF": RandomForestClassifier(
        n_estimators=500, max_features="sqrt", min_samples_leaf=1, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "ET": ExtraTreesClassifier(
        n_estimators=500, max_features="sqrt", min_samples_leaf=1, class_weight="balanced", random_state=42, n_jobs=-1
    )
}

weights = {"SVM": 0.40, "LR": 0.10, "RF": 0.30, "ET": 0.20}

print("Fitting individual models and computing test probabilities...")
test_probs = np.zeros((len(X_test), n_classes))

for m_name, clf in models.items():
    t_start = time.time()
    print(f"Training {m_name} (weight: {weights[m_name]:.2f})...", end=" ", flush=True)
    clf.fit(X_train, y_train)
    raw_probs = clf.predict_proba(X_test)

    c_list = clf.classes_ if hasattr(clf, "classes_") else clf.named_steps[list(clf.named_steps.keys())[-1]].classes_
    aligned = np.zeros((len(X_test), n_classes))
    for j, label in enumerate(c_list):
        pos = np.where(classes == label)[0][0]
        aligned[:, pos] = raw_probs[:, j]

    test_probs += weights[m_name] * aligned
    print(f"completed in {time.time()-t_start:.1f}s")

assert np.allclose(test_probs.sum(axis=1), 1.0, atol=1e-4)
print("Ensemble probability prediction complete.")

## 3. Formatting Submission & Integrity Verification

In [ ]:
# Rank top 3 classes per recording
top3_idx = np.argsort(test_probs, axis=1)[:, ::-1][:, :3]
top3_labels = classes[top3_idx]
pred_strings = [" ".join(row) for row in top3_labels]

pred_df = pd.DataFrame({"id": test_ids, "label": pred_strings})
sub = sample_df[["id"]].merge(pred_df, on="id", how="left")

# Rigorous validation checks
assert sub.shape == (1592, 2)
assert (sub["id"] == sample_df["id"]).all()
assert sub["label"].notna().all()
for lstr in sub["label"]:
    p = lstr.split()
    assert len(p) == 3 and len(set(p)) == 3 and all(c in classes for c in p)

# Write output submission file
sub.to_csv("submission.csv", index=False)
if WORK != Path("."):
    sub.to_csv(WORK / "submission.csv", index=False)

print("Saved submission.csv successfully!")
print(sub.head(10))